# ML-06 — Optional Full Signal Audit + Top-20 Skeptical Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadAyyanHassan/flyrank-ml-internship-work/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

**Lane:** Refresh / Content Opportunity Scoring  
**Decision window:** March 2026  
**Outcome window:** April 2026

This is the **optional deeper version** of the Week-4 baseline work. It does not replace the required `w04_baseline_score.ipynb`. It audits three decision-time signals, including the required FlyRank flag-linked CTR-vs-position signal, then applies the frozen baseline rule and performs a **top-20** skeptical review.

**Important boundary:** March observations are used to define and rank signals. April is used only after the decision-time analysis to measure the future outcome and assign an evidence verdict. No April field or future label is used to construct the rule.


## 1. Distributions — look before deciding

The audit starts with March-only distributions of impressions, CTR, average position, and impression-day coverage. Heavy tails and sparse coverage matter because a simple threshold can otherwise look more reliable than it is.


In [1]:
%pip -q install duckdb pandas numpy

import json
import os
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd
from IPython.display import display

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is missing. Add your Read token to Colab Secrets as HF_TOKEN.")

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute("CREATE OR REPLACE SECRET hf_token (TYPE huggingface, TOKEN ?)", [HF_TOKEN])

MARCH_REL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"
APRIL_REL = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-04/*.parquet"
OUTPUT_DIR = Path("work/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Warehouse connection configured.")
print("Decision window: March 2026")
print("Outcome window: April 2026")


Warehouse connection configured.
Decision window: March 2026
Outcome window: April 2026


In [2]:
schema = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{MARCH_REL}')").df()
required_columns = {
    "report_date", "client_hash_id", "content_hash_id", "gsc_data_available",
    "gsc_impressions", "gsc_clicks", "gsc_avg_position", "month"
}
missing = sorted(required_columns - set(schema["column_name"]))
assert not missing, f"Required warehouse columns are missing: {missing}"
display(schema[["column_name", "column_type"]])
print("PASS: required March warehouse fields are present.")


,column_name,column_type
0,report_date,DATE
1,client_hash_id,VARCHAR
2,content_hash_id,VARCHAR
3,client_has_gsc,BOOLEAN
4,client_has_ga4,BOOLEAN
5,gsc_data_available,BOOLEAN
6,ga4_data_available,BOOLEAN
7,gsc_impressions,BIGINT
8,gsc_clicks,BIGINT
9,gsc_sum_position,BIGINT


PASS: required March warehouse fields are present.


In [3]:
evaluation_sql = f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS march_impressions,
        SUM(gsc_clicks) AS march_clicks,
        CASE WHEN SUM(gsc_impressions) > 0
             THEN 100.0 * SUM(gsc_clicks) / SUM(gsc_impressions)
             ELSE NULL END AS march_ctr_pct,
        SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position > 0
                 THEN gsc_impressions * gsc_avg_position ELSE 0 END)
        / NULLIF(
            SUM(CASE WHEN gsc_impressions > 0 AND gsc_avg_position > 0
                     THEN gsc_impressions ELSE 0 END), 0
        ) AS march_avg_position,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END)
            AS march_impression_days
    FROM read_parquet('{MARCH_REL}')
    WHERE month = '2026-03' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),
april AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS april_impressions
    FROM read_parquet('{APRIL_REL}')
    WHERE month = '2026-04' AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    m.*,
    a.april_impressions,
    CASE WHEN m.march_impressions > 0
              AND a.april_impressions < 0.80 * m.march_impressions
         THEN 1 ELSE 0 END AS future_decline_label
FROM march m
INNER JOIN april a USING (client_hash_id, content_hash_id)
"""

evaluation_frame = con.execute(evaluation_sql).df()
assert evaluation_frame[["client_hash_id", "content_hash_id"]].duplicated().sum() == 0
base_rate = evaluation_frame["future_decline_label"].mean()

print(f"Decision/evaluation rows: {len(evaluation_frame):,}")
print(f"Future-decline base rate (evaluation only): {100 * base_rate:.2f}%")
display(evaluation_frame.head(10))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Decision/evaluation rows: 158,549
Future-decline base rate (evaluation only): 47.82%


,client_hash_id,content_hash_id,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_impression_days,april_impressions,future_decline_label
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,0.000000,4.742857,24,27.0,1
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,10849.0,22.0,0.202784,8.049866,31,16121.0,0
2,client_62f4a7e64f5e0096,content_e689bc511192751a,61.0,0.0,0.000000,6.410714,27,81.0,0
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,705.0,1.0,0.141844,5.872159,31,213.0,1
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,50.0,0.0,0.000000,15.276596,21,23.0,1
5,client_62f4a7e64f5e0096,content_df22bda1218f13ff,2099.0,1.0,0.047642,2.632206,31,1472.0,1
6,client_62f4a7e64f5e0096,content_aa184c1b4ea518e5,288.0,0.0,0.000000,10.526132,30,185.0,1
7,client_62f4a7e64f5e0096,content_b4de71c8ef5c4791,3535.0,25.0,0.707214,3.151344,31,10500.0,0
8,client_62f4a7e64f5e0096,content_d7568011c4325a33,1558.0,3.0,0.192555,5.247112,31,746.0,1
9,client_62f4a7e64f5e0096,content_e847a4dcc8af3742,2021.0,0.0,0.000000,7.853538,31,675.0,1


In [4]:
fields = ["march_impressions", "march_ctr_pct", "march_avg_position", "march_impression_days"]
distribution = pd.DataFrame({
    "field": fields,
    "n_non_null": [evaluation_frame[c].notna().sum() for c in fields],
    "median": [evaluation_frame[c].median() for c in fields],
    "p90": [evaluation_frame[c].quantile(0.90) for c in fields],
    "max": [evaluation_frame[c].max() for c in fields],
})
distribution[["median", "p90", "max"]] = distribution[["median", "p90", "max"]].round(3)
display(distribution)
print("Observation: impressions are heavy-tailed, so bucket views and the fixed threshold are reported rather than relying on a mean.")


,field,n_non_null,median,p90,max
0,march_impressions,158549,246.000,4427.20,617124.0
1,march_ctr_pct,158549,0.000,0.65,100.0
2,march_avg_position,157790,8.862,43.10,309.0
3,march_impression_days,158549,29.000,31.00,31.0


Observation: impressions are heavy-tailed, so bucket views and the fixed threshold are reported rather than relying on a mean.


## 2. Signal test #1 — Volume / quick-win

**Hypothesis:** pages with enough March search visibility are useful candidates for a practical content-review queue.

**Decision-time rule:** `march_impressions >= 500`.

The verdict is based on the difference between the tested group's future-decline rate and the overall base rate. A difference of at least 5 percentage points is treated as meaningful for this small audit.


In [5]:
signal1 = evaluation_frame.copy()
signal1["volume_bucket"] = pd.qcut(
    signal1["march_impressions"].rank(method="first"), q=5,
    labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"]
)
volume_buckets = (
    signal1.groupby("volume_bucket", observed=False)
    .agg(n=("future_decline_label", "size"),
         decline_rate=("future_decline_label", "mean"),
         median_march_impressions=("march_impressions", "median"))
    .reset_index()
)
volume_buckets["decline_rate_pct"] = (100 * volume_buckets["decline_rate"]).round(2)

volume_threshold = 500
high_volume = signal1["march_impressions"].ge(volume_threshold)
volume_n = int(high_volume.sum())
volume_rate = signal1.loc[high_volume, "future_decline_label"].mean()

if volume_n == 0:
    volume_verdict = "FALSE"
elif volume_rate >= base_rate + 0.05:
    volume_verdict = "CONFIRMED"
elif volume_rate <= base_rate - 0.05:
    volume_verdict = "OPPOSITE"
else:
    volume_verdict = "MIXED"

display(volume_buckets[["volume_bucket", "n", "decline_rate_pct", "median_march_impressions"]])
print(f"Fixed threshold: March impressions >= {volume_threshold:,}")
print(f"n above threshold: {volume_n:,}")
print(f"Decline rate above threshold: {100 * volume_rate:.2f}%")
print(f"Overall base rate: {100 * base_rate:.2f}%")
print(f"VERDICT: {volume_verdict}")


,volume_bucket,n,decline_rate_pct,median_march_impressions
0,Q1 lowest,31710,35.14,7.0
1,Q2,31710,49.90,61.0
2,Q3,31709,53.63,246.0
3,Q4,31710,53.13,895.0
4,Q5 highest,31710,47.28,4427.5


Fixed threshold: March impressions >= 500
n above threshold: 61,846
Decline rate above threshold: 49.99%
Overall base rate: 47.82%
VERDICT: MIXED


## 2. Signal test #2 — CTR-vs-position / CTR-fix **(FlyRank flag-linked)**

This is the required flag-linked audit. The visible band is positions **4–20** with at least **500 March impressions**, and the CTR cutoff is the **March-only median CTR within that band**.

The cutoff is calculated before any April outcome is inspected.


In [6]:
ctr_test = evaluation_frame.copy()
visible_band = (
    ctr_test["march_impressions"].ge(500)
    & ctr_test["march_avg_position"].between(4, 20, inclusive="both")
    & ctr_test["march_ctr_pct"].notna()
)
ctr_cutoff = ctr_test.loc[visible_band, "march_ctr_pct"].median()
assert pd.notna(ctr_cutoff)

band = ctr_test.loc[visible_band].copy()
band["ctr_position_bucket"] = pd.qcut(
    band["march_ctr_pct"].rank(method="first"), q=5,
    labels=["Q1 lowest CTR", "Q2", "Q3", "Q4", "Q5 highest CTR"]
)
ctr_buckets = (
    band.groupby("ctr_position_bucket", observed=False)
    .agg(n=("future_decline_label", "size"),
         decline_rate=("future_decline_label", "mean"),
         median_ctr_pct=("march_ctr_pct", "median"),
         median_position=("march_avg_position", "median"))
    .reset_index()
)
ctr_buckets["decline_rate_pct"] = (100 * ctr_buckets["decline_rate"]).round(2)

ctr_candidate = visible_band & ctr_test["march_ctr_pct"].lt(ctr_cutoff)
ctr_n = int(ctr_candidate.sum())
ctr_rate = ctr_test.loc[ctr_candidate, "future_decline_label"].mean()

if ctr_n == 0:
    ctr_verdict = "FALSE"
elif ctr_rate >= base_rate + 0.05:
    ctr_verdict = "CONFIRMED"
elif ctr_rate <= base_rate - 0.05:
    ctr_verdict = "OPPOSITE"
else:
    ctr_verdict = "MIXED"

display(ctr_buckets[[
    "ctr_position_bucket", "n", "decline_rate_pct",
    "median_ctr_pct", "median_position"
]])
print("Visible position band: positions 4–20 with >=500 March impressions")
print(f"March-only CTR cutoff (median): {ctr_cutoff:.3f}%")
print(f"n below CTR cutoff: {ctr_n:,}")
print(f"Decline rate below cutoff: {100 * ctr_rate:.2f}%")
print(f"Overall base rate: {100 * base_rate:.2f}%")
print(f"VERDICT: {ctr_verdict}")


,ctr_position_bucket,n,decline_rate_pct,median_ctr_pct,median_position
0,Q1 lowest CTR,7252,63.97,0.000000,8.313263
1,Q2,7251,57.95,0.100220,7.216532
2,Q3,7251,51.55,0.190840,6.990895
3,Q4,7251,38.96,0.348584,6.490494
4,Q5 highest CTR,7251,32.27,0.676745,6.970395


Visible position band: positions 4–20 with >=500 March impressions
March-only CTR cutoff (median): 0.191%
n below CTR cutoff: 18,120
Decline rate below cutoff: 59.54%
Overall base rate: 47.82%
VERDICT: CONFIRMED


## 2. Signal test #3 — March impression-day coverage

**Hypothesis:** pages observed with impressions on more days in March provide a more stable decision-time signal than pages with very sparse visibility.

This is a **coverage-consistency proxy**, not a claim about the content publication date or literal page staleness.


In [7]:
signal3 = evaluation_frame.copy()
signal3["coverage_bucket"] = pd.qcut(
    signal3["march_impression_days"].rank(method="first"), q=5,
    labels=["Q1 lowest coverage", "Q2", "Q3", "Q4", "Q5 highest coverage"]
)
coverage_buckets = (
    signal3.groupby("coverage_bucket", observed=False)
    .agg(n=("future_decline_label", "size"),
         decline_rate=("future_decline_label", "mean"),
         median_impression_days=("march_impression_days", "median"))
    .reset_index()
)
coverage_buckets["decline_rate_pct"] = (100 * coverage_buckets["decline_rate"]).round(2)

coverage_cutoff = float(signal3["march_impression_days"].median())
coverage_candidate = signal3["march_impression_days"].ge(coverage_cutoff)
coverage_n = int(coverage_candidate.sum())
coverage_rate = signal3.loc[coverage_candidate, "future_decline_label"].mean()

if coverage_n == 0:
    coverage_verdict = "FALSE"
elif coverage_rate >= base_rate + 0.05:
    coverage_verdict = "CONFIRMED"
elif coverage_rate <= base_rate - 0.05:
    coverage_verdict = "OPPOSITE"
else:
    coverage_verdict = "MIXED"

display(coverage_buckets[[
    "coverage_bucket", "n", "decline_rate_pct", "median_impression_days"
]])
print(f"March-only coverage threshold: >= {coverage_cutoff:.1f} impression days")
print(f"n at/above coverage threshold: {coverage_n:,}")
print(f"Decline rate at/above threshold: {100 * coverage_rate:.2f}%")
print(f"Overall base rate: {100 * base_rate:.2f}%")
print(f"VERDICT: {coverage_verdict}")


,coverage_bucket,n,decline_rate_pct,median_impression_days
0,Q1 lowest coverage,31710,31.77,4.0
1,Q2,31710,43.67,17.0
2,Q3,31709,49.86,29.0
3,Q4,31710,61.20,31.0
4,Q5 highest coverage,31710,52.59,31.0


March-only coverage threshold: >= 29.0 impression days
n at/above coverage threshold: 80,098
Decline rate at/above threshold: 54.91%
Overall base rate: 47.82%
VERDICT: CONFIRMED


### Signal audit verdict summary

`CONFIRMED` means the tested group is at least 5 percentage points above the observed base rate in this development window; `OPPOSITE` means at least 5 points below; `MIXED` means the observed difference is smaller; `FALSE` means there were no usable rows.

A negative verdict is a useful result because it prevents a weak signal from becoming an unjustified rule feature.


In [8]:
signal_summary = pd.DataFrame([
    {"signal": "Volume / quick-win", "flag_linked": False,
     "verdict": volume_verdict, "tested_n": volume_n, "tested_rate": volume_rate},
    {"signal": "CTR-vs-position / CTR-fix", "flag_linked": True,
     "verdict": ctr_verdict, "tested_n": ctr_n, "tested_rate": ctr_rate},
    {"signal": "Impression-day coverage proxy", "flag_linked": False,
     "verdict": coverage_verdict, "tested_n": coverage_n, "tested_rate": coverage_rate},
])
signal_summary["tested_rate_pct"] = (100 * signal_summary["tested_rate"]).round(2)

display(signal_summary[["signal", "flag_linked", "verdict", "tested_n", "tested_rate_pct"]])
assert signal_summary["tested_n"].gt(0).all()
assert bool(signal_summary.loc[
    signal_summary["signal"].eq("CTR-vs-position / CTR-fix"), "flag_linked"
].iloc[0])
print("PASS: three signal tests have visible bucket tables and n; at least one is explicitly flag-linked.")


,signal,flag_linked,verdict,tested_n,tested_rate_pct
0,Volume / quick-win,False,MIXED,61846,49.99
1,CTR-vs-position / CTR-fix,True,CONFIRMED,18120,59.54
2,Impression-day coverage proxy,False,CONFIRMED,80098,54.91


PASS: three signal tests have visible bucket tables and n; at least one is explicitly flag-linked.


## 3. Frozen baseline rule + optional top-20 review

For comparability with the required notebook, the **same March-only baseline rule** is used:

- high volume = March impressions ≥ 500
- CTR-fix = high volume + March position 4–20 + March CTR below the March-only median CTR in that band
- score = 5 when both fire, 3 for high volume only, 0 otherwise

The optional work adds a **top-20** skeptical review. April is not used to justify any row.


In [9]:
scored = evaluation_frame[[
    "client_hash_id", "content_hash_id", "march_impressions",
    "march_clicks", "march_ctr_pct", "march_avg_position",
    "march_impression_days"
]].copy()

scored["high_volume"] = scored["march_impressions"].ge(volume_threshold)
scored["ctr_fix"] = (
    scored["high_volume"]
    & scored["march_avg_position"].between(4, 20, inclusive="both")
    & scored["march_ctr_pct"].notna()
    & scored["march_ctr_pct"].lt(ctr_cutoff)
)

scored["score"] = np.select(
    [scored["high_volume"] & scored["ctr_fix"], scored["high_volume"]],
    [5, 3], default=0
).astype(int)

scored["reason_code"] = np.select(
    [scored["high_volume"] & scored["ctr_fix"], scored["high_volume"]],
    ["high_volume_ctr_fix", "high_volume_only"], default="monitor"
)

scored["action"] = np.select(
    [scored["reason_code"].eq("high_volume_ctr_fix"),
     scored["reason_code"].eq("high_volume_only")],
    ["CTR-fix review", "Quick-win review"], default="Monitor"
)

ranked_queue = (
    scored.sort_values(
        ["score", "march_impressions", "march_clicks"],
        ascending=[False, False, False], kind="mergesort"
    ).reset_index(drop=True)
)
ranked_queue.insert(0, "rank", np.arange(1, len(ranked_queue) + 1))

queue_columns = [
    "rank", "client_hash_id", "content_hash_id", "score",
    "reason_code", "action", "march_impressions", "march_clicks",
    "march_ctr_pct", "march_avg_position", "march_impression_days"
]
ranked_queue = ranked_queue[queue_columns]

print(f"Ranked queue rows: {len(ranked_queue):,}")
display(ranked_queue.head(20))


Ranked queue rows: 158,549


,rank,client_hash_id,content_hash_id,score,reason_code,action,march_impressions,march_clicks,march_ctr_pct,march_avg_position,march_impression_days
0,1,client_62f4a7e64f5e0096,content_b99ea6861864dea5,5,high_volume_ctr_fix,CTR-fix review,194337.0,361.0,0.185760,4.551516,31
1,2,client_73cda7b4e4f265ea,content_f43118e089ecc69a,5,high_volume_ctr_fix,CTR-fix review,139417.0,191.0,0.136999,5.342218,31
2,3,client_62f4a7e64f5e0096,content_7c6373141eae744a,5,high_volume_ctr_fix,CTR-fix review,132593.0,83.0,0.062598,5.948459,31
3,4,client_73cda7b4e4f265ea,content_95ff62babbfac9c7,5,high_volume_ctr_fix,CTR-fix review,122857.0,234.0,0.190465,4.623098,31
4,5,client_23a62021009f63c4,content_5e1c049f62e33b11,5,high_volume_ctr_fix,CTR-fix review,120175.0,168.0,0.139796,17.773364,31
5,6,client_73cda7b4e4f265ea,content_e578ac84778da489,5,high_volume_ctr_fix,CTR-fix review,117764.0,163.0,0.138412,4.203950,31
6,7,client_62f4a7e64f5e0096,content_f6116743b00afc2d,5,high_volume_ctr_fix,CTR-fix review,107584.0,15.0,0.013943,9.735658,31
7,8,client_e547b89c05043229,content_21309e9a83c83653,5,high_volume_ctr_fix,CTR-fix review,103187.0,192.0,0.186070,5.036739,29
8,9,client_73cda7b4e4f265ea,content_cf651123f1085418,5,high_volume_ctr_fix,CTR-fix review,101363.0,175.0,0.172647,6.286308,31
9,10,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,5,high_volume_ctr_fix,CTR-fix review,89332.0,4.0,0.004478,7.831807,31


In [10]:
top20 = ranked_queue.head(20).copy()

why_map = {
    "high_volume_ctr_fix": "Both high-volume and CTR-fix conditions fired.",
    "high_volume_only": "March impressions reached the 500+ visibility threshold, but the CTR-fix condition did not fire.",
    "monitor": "No baseline action-priority condition fired.",
}
wrong_map = {
    "high_volume_ctr_fix": "CTR may be low for a legitimate reason at this position, or the page may already match search intent well.",
    "high_volume_only": "High visibility alone does not prove a content problem; the page may already perform appropriately for its position.",
    "monitor": "The simple baseline can miss a real opportunity because it uses only two action signals.",
}

top20["why_its_here"] = top20["reason_code"].map(why_map)
top20["what_would_make_it_wrong"] = top20["reason_code"].map(wrong_map)

display(top20[[
    "rank", "client_hash_id", "content_hash_id", "action",
    "why_its_here", "what_would_make_it_wrong"
]])
assert len(top20) == min(20, len(ranked_queue))
print(f"Top-20 review rows: {len(top20)}")


,rank,client_hash_id,content_hash_id,action,why_its_here,what_would_make_it_wrong
0,1,client_62f4a7e64f5e0096,content_b99ea6861864dea5,CTR-fix review,Both high-volume and CTR-fix conditions fired.,CTR may be low for a legitimate reason at this...
1,2,client_73cda7b4e4f265ea,content_f43118e089ecc69a,CTR-fix review,Both high-volume and CTR-fix conditions fired.,CTR may be low for a legitimate reason at this...
2,3,client_62f4a7e64f5e0096,content_7c6373141eae744a,CTR-fix review,Both high-volume and CTR-fix conditions fired.,CTR may be low for a legitimate reason at this...
3,4,client_73cda7b4e4f265ea,content_95ff62babbfac9c7,CTR-fix review,Both high-volume and CTR-fix conditions fired.,CTR may be low for a legitimate reason at this...
4,5,client_23a62021009f63c4,content_5e1c049f62e33b11,CTR-fix review,Both high-volume and CTR-fix conditions fired.,CTR may be low for a legitimate reason at this...
5,6,client_73cda7b4e4f265ea,content_e578ac84778da489,CTR-fix review,Both high-volume and CTR-fix conditions fired.,CTR may be low for a legitimate reason at this...
6,7,client_62f4a7e64f5e0096,content_f6116743b00afc2d,CTR-fix review,Both high-volume and CTR-fix conditions fired.,CTR may be low for a legitimate reason at this...
7,8,client_e547b89c05043229,content_21309e9a83c83653,CTR-fix review,Both high-volume and CTR-fix conditions fired.,CTR may be low for a legitimate reason at this...
8,9,client_73cda7b4e4f265ea,content_cf651123f1085418,CTR-fix review,Both high-volume and CTR-fix conditions fired.,CTR may be low for a legitimate reason at this...
9,10,client_9958f0a7ae1df715,content_cd3d932d4e1c8db0,CTR-fix review,Both high-volume and CTR-fix conditions fired.,CTR may be low for a legitimate reason at this...


Top-20 review rows: 20


## 4. Weak picks + post-hoc outcome check

A weak pick is a row whose rationale is easy to challenge even though the score is internally correct. The first weak candidate is selected from `high_volume_only` when available because volume alone is weaker evidence than the combined rule.

The post-hoc outcome check is a diagnostic only.


In [11]:
weak_candidates = ranked_queue[
    ranked_queue["reason_code"].eq("high_volume_only")
].copy()

if len(weak_candidates):
    weak_pick = weak_candidates.head(1).copy()
    weak_pick["weakness"] = (
        "High volume alone is a weak rationale: the page has visibility, "
        "but the CTR-vs-position condition did not fire."
    )
else:
    weak_pick = (
        ranked_queue[ranked_queue["reason_code"].eq("high_volume_ctr_fix")]
        .sort_values("march_ctr_pct", ascending=False)
        .head(1).copy()
    )
    weak_pick["weakness"] = (
        "Borderline combined-signal pick: its CTR is closest to the "
        "March-only cutoff among the combined-signal items."
    )

display(weak_pick[[
    "rank", "client_hash_id", "content_hash_id", "score",
    "reason_code", "action", "march_impressions",
    "march_ctr_pct", "march_avg_position", "weakness"
]])

evaluation_lookup = evaluation_frame[
    ["client_hash_id", "content_hash_id", "future_decline_label"]
]
ranked_with_outcome = ranked_queue.merge(
    evaluation_lookup,
    on=["client_hash_id", "content_hash_id"],
    how="left", validate="one_to_one"
)
top20_posthoc = ranked_with_outcome.head(20)
false_positive_count = int(top20_posthoc["future_decline_label"].eq(0).sum())
print(f"Post-hoc false positives in top 20: {false_positive_count} / {len(top20_posthoc)}")


,rank,client_hash_id,content_hash_id,score,reason_code,action,march_impressions,march_ctr_pct,march_avg_position,weakness
18120,18121,client_e547b89c05043229,content_eadb33b5df496f4a,3,high_volume_only,Quick-win review,617124.0,0.918454,2.33147,High volume alone is a weak rationale: the pag...


Post-hoc false positives in top 20: 10 / 20


## 5. Leakage / integrity self-check

The scored queue must contain only March decision-time fields plus the score, reason code, and action. April impressions and the future label are explicitly forbidden from the queue.


In [12]:
forbidden_queue_columns = {
    "april_impressions", "future_decline_label", "trend_pct",
    "trend_direction", "is_declining", "leaked_decline_score"
}
assert forbidden_queue_columns.isdisjoint(set(ranked_queue.columns))
assert "client_name" not in ranked_queue.columns
assert "url" not in ranked_queue.columns
assert "query" not in ranked_queue.columns
assert len(signal_summary) == 3
assert signal_summary["tested_n"].gt(0).all()

print("PASS: no future-window or label-derived fields are present in the scored queue.")
print("PASS: three signal audits completed; CTR-vs-position is flag-linked.")
print("PASS: top-20 skeptical review completed.")


PASS: no future-window or label-derived fields are present in the scored queue.
PASS: three signal audits completed; CTR-vs-position is flag-linked.
PASS: top-20 skeptical review completed.


In [13]:
audit_receipt = {
    "lane": "Refresh / Content Opportunity Scoring",
    "decision_window": "2026-03",
    "outcome_window": "2026-04",
    "decision_rows": int(len(evaluation_frame)),
    "future_decline_rate": float(base_rate),
    "signal_verdicts": {
        "volume": volume_verdict,
        "ctr_vs_position_flag_linked": ctr_verdict,
        "impression_day_coverage_proxy": coverage_verdict,
    },
    "top20_review_rows": int(len(top20)),
    "top20_posthoc_false_positives": false_positive_count,
    "queue_columns": queue_columns,
    "leakage_guard": "PASS",
}
receipt_path = OUTPUT_DIR / "signal_audit_metrics.json"
receipt_path.write_text(json.dumps(audit_receipt, indent=2), encoding="utf-8")
print(f"Metrics receipt written to: {receipt_path}")


Metrics receipt written to: work/outputs/signal_audit_metrics.json


## Final self-check

- [ ] Three signal tests completed with visible bucket tables and `n`.
- [ ] At least one signal is explicitly linked to a real FlyRank flag: CTR-vs-position / CTR-fix.
- [ ] Signal cutoffs are derived from March only.
- [ ] The coverage signal is described honestly as a proxy, not as literal content staleness.
- [ ] The same frozen baseline rule is used for the optional top-20 review.
- [ ] Each top-20 row has an action, why it is there, and what would make it wrong.
- [ ] April/future outcome is used only for post-hoc evaluation.
- [ ] No future-window or label-derived fields are present in the scored queue.
- [ ] `work/outputs/signal_audit_metrics.json` is generated as a run receipt.
- [ ] Run **Runtime → Run all** in Colab with the real `HF_TOKEN`, inspect every output, then commit the notebook under `work/notebooks/w04_signal_audit.ipynb`.

**Submission strategy:** the required card still asks for the repo URL. This optional notebook strengthens the repo evidence; it does not replace the required executed baseline notebook.
